In [1]:
import torch
from torch import nn
import numpy as np
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# -- DATA ORGANIZATION --

train = pd.read_csv("data/train.csv")
val = pd.read_csv("data/val.csv")
test = pd.read_csv("data/test.csv")

feature_cols = ['qoq_rev', 'yoy_rev', 'qoq_eps', 'yoy_eps', 'momentum', 
                'net_margin', 'gross_margin', 'oper_margin', 'roe',
                'de_ratio', 'current_ratio', 'cash_ratio', 'asset_turn']

In [17]:
X_train = torch.FloatTensor(train[feature_cols].values)
X_val = torch.FloatTensor(val[feature_cols].values)
X_test = torch.FloatTensor(test[feature_cols].values)

y_train = torch.FloatTensor(train['eps_status'].values)
y_val = torch.FloatTensor(val['eps_status'].values)
y_test = torch.FloatTensor(test['eps_status'].values)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [18]:
# -- MODEL: SIMPLE FEEDFORWARD CLASSIFIER --

class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Linear(in_features=13, out_features=16),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(in_features=16, out_features=8),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(in_features=8, out_features=1)
        )
    
    def forward(self, x):
        return self.sequential(x)

    
# -- METRICS--

def accuracy_fn(y_true, y_pred):
  correct = torch.eq(y_true, y_pred).sum().item()
  acc = (correct / len(y_pred)) * 100
  return acc

# -- MODEL, LOSS AND OPTIMIZER

modelv1 = Classifier()

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(lr=0.001, params=modelv1.parameters())

In [19]:
# -- TRAINING LOOP --

SEED = 35
epochs = 400

torch.manual_seed(SEED)


for epoch in range(epochs):
    modelv1.train()
    for x, y in train_loader:
        y_logits = modelv1(x).squeeze(1)
        y_preds = torch.round(torch.sigmoid(y_logits))

        loss = loss_fn(y_logits, y)
        acc = accuracy_fn(y_true=y, y_pred=y_preds)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    modelv1.eval()
    with torch.inference_mode():
        for x_val, y_val in val_loader:
            val_logits = modelv1(x_val).squeeze(1)
            val_preds = torch.round(torch.sigmoid(val_logits))

            val_loss = loss_fn(val_logits, y_val)
            val_acc = accuracy_fn(y_true=y_val, y_pred=val_preds)

    if epoch % 50 == 0:
        print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Val loss: {val_loss:.5f}, Val acc: {val_acc:.2f}%")


total_params = sum(p.numel() for p in modelv1.parameters())
print(f"Total parameters: {total_params}")

Epoch: 0 | Loss: 6.94217, Accuracy: 65.00% | Val loss: 1.29222, Val acc: 55.00%
Epoch: 50 | Loss: 0.90210, Accuracy: 57.50% | Val loss: 0.63439, Val acc: 70.00%
Epoch: 100 | Loss: 0.63333, Accuracy: 67.50% | Val loss: 0.58887, Val acc: 70.00%
Epoch: 150 | Loss: 0.92368, Accuracy: 70.00% | Val loss: 0.62356, Val acc: 65.00%
Epoch: 200 | Loss: 0.65087, Accuracy: 65.00% | Val loss: 0.66223, Val acc: 60.00%
Epoch: 250 | Loss: 0.55504, Accuracy: 72.50% | Val loss: 0.58594, Val acc: 75.00%
Epoch: 300 | Loss: 0.49363, Accuracy: 80.00% | Val loss: 0.69327, Val acc: 55.00%
Epoch: 350 | Loss: 0.57108, Accuracy: 72.50% | Val loss: 0.47658, Val acc: 90.00%
Total parameters: 369


In [20]:
# -- TEST -- 

for x, y in test_loader:
    test_logits = modelv1(x).squeeze(1)
    test_preds = torch.round(torch.sigmoid(test_logits))

    test_loss = loss_fn(test_logits, y)
    test_acc = accuracy_fn(y_true=y, y_pred=test_preds)

    print(f"Loss: {test_loss:.2f} | Accuracy {test_acc:.2f}%")

Loss: 0.57 | Accuracy 70.31%
Loss: 0.53 | Accuracy 73.44%
Loss: 0.68 | Accuracy 62.50%
Loss: 0.62 | Accuracy 65.00%
